# Notebook 03 — Modelos Baseline

## Objetivo

Entrenar modelos clásicos de clasificación supervisada sobre representaciones BoW y TF-IDF.

**Experimento principal:** subconjunto político.  
**Experimento complementario:** dataset completo (solo baselines).

> El modelo aprende patrones lingüísticos del dataset; no verifica hechos.

**Métrica principal:** F2-score de la clase fake (prioriza recall de fake sin ignorar precisión).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.paths import *
from src.plotting import setup_style, save_figure

setup_style()

from itertools import product
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, roc_curve, auc
import joblib
from tqdm.auto import tqdm

from src.metrics import compute_metrics, metrics_to_row
from src.modeling import MODEL_CONFIGS, TEXT_FIELDS, STOPWORD_OPTS, build_model, build_vectorizer, get_text_col
from src.plotting import plot_confusion_matrix


## 1. Carga de splits

In [2]:

def load_splits(prefix):
    train = pd.read_csv(DATA_PROCESSED / f'{prefix}_train.csv')
    val = pd.read_csv(DATA_PROCESSED / f'{prefix}_val.csv')
    test = pd.read_csv(DATA_PROCESSED / f'{prefix}_test.csv')
    return train, val, test

pol_train, pol_val, pol_test = load_splits('politics')
full_train, full_val, full_test = load_splits('full')


## 2. Configuración de la grilla de experimentos

In [3]:

NGRAM_OPTS = [(1, 1), (1, 2)]
MAX_FEATURES_OPTS = [10000, 30000, 50000]
VECTORIZER_TYPES = ['bow', 'tfidf']


## 3. Entrenamiento y selección por validación (F2 fake)

In [4]:

def run_baseline_grid(train, val, test, dataset_scope, max_combos=None):
    results = []
    combos = list(product(
        TEXT_FIELDS.keys(), STOPWORD_OPTS.keys(), NGRAM_OPTS,
        MAX_FEATURES_OPTS, VECTORIZER_TYPES, MODEL_CONFIGS.keys()
    ))
    if max_combos:
        combos = combos[:max_combos]

    for field, stop, ngram, max_feat, vtype, model_name in tqdm(combos, desc=dataset_scope):
        text_col = get_text_col(field, stop)
        X_train = train[text_col].fillna('')
        X_val = val[text_col].fillna('')
        X_test = test[text_col].fillna('')
        y_train, y_val, y_test = train['label'], val['label'], test['label']

        cfg = MODEL_CONFIGS[model_name]
        param_name = cfg['param_name']
        best_f2, best_pipe, best_param = -1, None, None

        for param_val in cfg['params'][param_name]:
            pipe = Pipeline([
                ('vec', build_vectorizer(vtype, ngram, max_feat)),
                ('clf', build_model(model_name, param_val)),
            ])
            pipe.fit(X_train, y_train)
            y_val_pred = pipe.predict(X_val)
            m = compute_metrics(y_val, y_val_pred)
            if m['f2_fake'] > best_f2:
                best_f2 = m['f2_fake']
                best_pipe = pipe
                best_param = param_val

        y_test_pred = best_pipe.predict(X_test)
        y_proba = None
        if hasattr(best_pipe.named_steps['clf'], 'predict_proba'):
            y_proba = best_pipe.predict_proba(X_test)[:, 1]
        elif hasattr(best_pipe.named_steps['clf'], 'decision_function'):
            scores = best_pipe.decision_function(X_test)
            y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

        test_m = compute_metrics(y_test, y_test_pred, y_proba)
        row = metrics_to_row(test_m, {
            'model': model_name,
            'vectorizer': vtype,
            'text_field': field,
            'stopwords': stop,
            'ngram_range': str(ngram),
            'max_features': max_feat,
            'best_param': best_param,
            'dataset_scope': dataset_scope,
            'split': 'test',
        })
        results.append(row)

    return pd.DataFrame(results)

# Experimento principal: subconjunto político
politics_results = run_baseline_grid(pol_train, pol_val, pol_test, 'politics')


politics:   0%|          | 0/216 [00:00<?, ?it/s]

In [6]:

# Experimento complementario: dataset completo
full_results = run_baseline_grid(full_train, full_val, full_test, 'full_dataset')
baseline_results = pd.concat([politics_results, full_results], ignore_index=True)
baseline_results = baseline_results.sort_values('f2_fake', ascending=False)
baseline_results.to_csv(RESULTS_METRICS / 'baseline_results.csv', index=False)
print('Mejores 10 modelos:')
baseline_results.head(10)


full_dataset:   0%|          | 0/216 [00:00<?, ?it/s]

Mejores 10 modelos:


,accuracy,precision_fake,recall_fake,f1_fake,f2_fake,roc_auc,model,vectorizer,text_field,stopwords,ngram_range,max_features,best_param,dataset_scope,split
204,0.996307,0.995556,0.993348,0.994451,0.993789,0.999001,logistic_regression,bow,full,without_stopwords,"(1, 2)",30000,1.0,politics,test
198,0.996307,0.995556,0.993348,0.994451,0.993789,0.998836,logistic_regression,bow,full,without_stopwords,"(1, 2)",10000,0.1,politics,test
210,0.995938,0.995551,0.992239,0.993892,0.992900,0.998917,logistic_regression,bow,full,without_stopwords,"(1, 2)",50000,1.0,politics,test
200,0.995569,0.994444,0.992239,0.993341,0.992680,0.998218,linear_svm,bow,full,without_stopwords,"(1, 2)",10000,0.1,politics,test
146,0.994830,0.992239,0.992239,0.992239,0.992239,0.998589,linear_svm,bow,full,with_stopwords,"(1, 1)",10000,0.1,politics,test
170,0.995569,0.995546,0.991131,0.993333,0.992011,0.999145,linear_svm,bow,full,with_stopwords,"(1, 2)",30000,0.1,politics,test
92,0.995199,0.994438,0.991131,0.992782,0.991791,0.998791,linear_svm,bow,body,with_stopwords,"(1, 2)",10000,1.0,politics,test
116,0.995199,0.994438,0.991131,0.992782,0.991791,0.997661,linear_svm,bow,body,without_stopwords,"(1, 1)",30000,0.1,politics,test
428,0.997272,0.982353,0.994048,0.988166,0.991686,0.998811,linear_svm,bow,full,without_stopwords,"(1, 2)",50000,0.1,full_dataset,test
128,0.994830,0.993333,0.991131,0.992231,0.991571,0.997985,linear_svm,bow,body,without_stopwords,"(1, 2)",10000,0.1,politics,test


## 4. Mejor modelo y evaluación final

In [7]:

best_row = baseline_results.iloc[0]
print('Mejor modelo (test):')
print(best_row.to_string())

# Re-entrenar mejor config en politics para guardar
scope = best_row['dataset_scope']
if scope == 'politics':
    tr, va, te = pol_train, pol_val, pol_test
else:
    tr, va, te = full_train, full_val, full_test

text_col = get_text_col(best_row['text_field'], best_row['stopwords'])
ngram = eval(best_row['ngram_range'])
pipe = Pipeline([
    ('vec', build_vectorizer(best_row['vectorizer'], ngram, int(best_row['max_features']))),
    ('clf', build_model(best_row['model'], best_row['best_param'])),
])
pipe.fit(tr[text_col].fillna(''), tr['label'])
y_pred = pipe.predict(te[text_col].fillna(''))
cm = confusion_matrix(te['label'], y_pred)

plot_confusion_matrix(cm, ['Real', 'Fake'], f'Matriz de confusión — {best_row["model"]}',
                       RESULTS_FIGURES / 'baseline_best_confusion_matrix.png')

# Curva ROC
if hasattr(pipe.named_steps['clf'], 'predict_proba'):
    y_proba = pipe.predict_proba(te[text_col].fillna(''))[:, 1]
elif hasattr(pipe.named_steps['clf'], 'decision_function'):
    scores = pipe.decision_function(te[text_col].fillna(''))
    y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
else:
    y_proba = None

if y_proba is not None:
    fpr, tpr, _ = roc_curve(te['label'], y_proba)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('Curva ROC — mejor baseline')
    ax.legend()
    save_figure(fig, RESULTS_FIGURES / 'baseline_best_roc_curve.png')
    plt.show()

joblib.dump(pipe, RESULTS_MODELS / 'best_baseline_model.joblib')
print('Modelo guardado en results/models/')

# Guardar siempre el mejor modelo del subconjunto político (experimento principal)
pol_best = politics_results.sort_values('f2_fake', ascending=False).iloc[0]
text_col_p = get_text_col(pol_best['text_field'], pol_best['stopwords'])
ngram_p = eval(pol_best['ngram_range'])
pipe_pol = Pipeline([
    ('vec', build_vectorizer(pol_best['vectorizer'], ngram_p, int(pol_best['max_features']))),
    ('clf', build_model(pol_best['model'], pol_best['best_param'])),
])
pipe_pol.fit(pol_train[text_col_p].fillna(''), pol_train['label'])
joblib.dump(pipe_pol, RESULTS_MODELS / 'best_baseline_politics.joblib')
pol_best.to_json(RESULTS_MODELS / 'best_baseline_politics_config.json')
print('Mejor modelo politics guardado: best_baseline_politics.joblib')


Mejor modelo (test):
accuracy                     0.996307
precision_fake               0.995556
recall_fake                  0.993348
f1_fake                      0.994451
f2_fake                      0.993789
roc_auc                      0.999001
model             logistic_regression
vectorizer                        bow
text_field                       full
stopwords           without_stopwords
ngram_range                    (1, 2)
max_features                    30000
best_param                        1.0
dataset_scope                politics
split                            test
Modelo guardado en results/models/
Mejor modelo politics guardado: best_baseline_politics.joblib


## 5. Comparación politics vs full_dataset

Si el rendimiento en el dataset completo es mucho mayor que en el subconjunto político, probablemente parte del rendimiento se explica por sesgos de tema, fuente o estructura del dataset — no solo por patrones lingüísticos de fake news.

In [8]:

compare = baseline_results.groupby('dataset_scope')[['f2_fake', 'accuracy', 'recall_fake', 'precision_fake']].max()
print('Mejores métricas por alcance:')
print(compare)

fig, ax = plt.subplots(figsize=(8, 4))
top_pol = politics_results.nlargest(5, 'f2_fake')['f2_fake'].mean()
top_full = full_results.nlargest(5, 'f2_fake')['f2_fake'].mean()
sns.barplot(x=['politics (top-5 avg)', 'full_dataset (top-5 avg)'], y=[top_pol, top_full], ax=ax)
ax.set_ylabel('F2 fake (promedio top-5)')
ax.set_title('Comparación subconjunto político vs dataset completo')
save_figure(fig, RESULTS_FIGURES / 'baseline_politics_vs_full.png')
plt.show()


Mejores métricas por alcance:
                f2_fake  accuracy  recall_fake  precision_fake
dataset_scope                                                 
full_dataset   0.991686  0.997783     0.994048        0.989599
politics       0.993789  0.996307     0.993348        0.996652


## Conclusiones

- Se compararon LR, MNB y Linear SVM con BoW y TF-IDF.
- Se evaluaron título, cuerpo y título+cuerpo; con y sin stopwords.
- La métrica principal F2 prioriza detectar fake news (minimizar falsos negativos).
- El modelo aprende patrones del dataset, no verifica hechos.